In [ ]:
"""
=============================================================================
  Raw-Data Artifact Analysis — VP004, EEG + fNIRS (BEFORE preprocessing)
=============================================================================

Purpose
-------
The preprocessed-data analysis found that VP004 is by far the worst-quality
subject in EEG (artifact score 80-83/100 in all 3 sessions) but only
mildly noisy in fNIRS (score 8-12/100). This script checks whether that
EEG-specific problem is already present in the RAW data (before ICA,
filtering, or any other preprocessing step) — which would indicate a
genuine physiological/recording issue (e.g. poor electrode contact,
muscle tension) rather than an artifact introduced by preprocessing itself.

Dataset format (Shin et al., 2018 — BBCI n-back EEG-fNIRS dataset)
--------------------------------------------------------------------
  {BASE_DATASET_PATH}\\{subject_id}-EEG\\cnt_nback.mat   -> raw EEG signal + markers
  {BASE_DATASET_PATH}\\{subject_id}-EEG\\mrk_nback.mat   -> EEG trial markers
  {BASE_DATASET_PATH}\\{subject_id}-EEG\\mnt_nback.mat   -> EEG electrode positions
  {BASE_DATASET_PATH}\\{subject_id}-NIRS\\cnt_nback.mat  -> raw fNIRS oxy/deoxy signal
  {BASE_DATASET_PATH}\\{subject_id}-NIRS\\mrk_nback.mat  -> fNIRS trial markers
  {BASE_DATASET_PATH}\\{subject_id}-NIRS\\mnt_nback.mat  -> fNIRS channel positions

  cnt_nback.mat  (EEG)   -> struct with fields: .x (n_samples, n_channels),
                            .fs (sampling rate), .clab (channel labels)
  cnt_nback.mat  (fNIRS) -> struct with fields: .oxy.x, .deoxy.x
                            (n_samples, n_channels), .oxy.fs / .deoxy.fs

This script loads the RAW .mat files directly (bypassing every
preprocessing step: no filtering, no ICA, no resampling, no rereferencing)
and runs the same artifact-detection battery used for the preprocessed
data, so results are directly comparable.

Output
------
  - One text report + one figure for VP004's raw EEG (whole recording,
    all sessions together, since raw .mat files are not pre-split by
    session in this dataset the way the cleaned .fif files are)
  - One text report + one figure for VP004's raw fNIRS (HbO/HbR)
  - A short comparison summary: raw vs preprocessed quality scores

Install:
  pip install numpy scipy matplotlib pywavelets
=============================================================================
"""

import os
import warnings
import numpy as np
import scipy.io as sio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings("ignore")

from scipy import signal, stats
from scipy.signal import welch, butter, filtfilt, find_peaks
import pywt

# ══════════════════════════════════════════════════════════════════════════
#  0. Configuration
# ══════════════════════════════════════════════════════════════════════════

BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_DIR         = r"E:\kntu\Thesis\codes\Results\raw_data_artifact_report"

SUBJECT_ID = "VP004"

# EEG thresholds (microvolts, µV) — same values used in the preprocessed-data script
THRESH_ZSCORE       = 3.5
THRESH_KURTOSIS     = 5.0
THRESH_AMP_UV       = 150
THRESH_EMG_RATIO    = 2.5
CONTAM_FRAC_THRESH  = 0.01   # fraction-based flagging, same fix as before

# fNIRS thresholds
THRESH_FNIRS_DRIFT    = 0.5
THRESH_FNIRS_HBR_CORR = 0.1
THRESH_FNIRS_SNR_DB   = 3.0
THRESH_FNIRS_KURTOSIS = 5.0


# ══════════════════════════════════════════════════════════════════════════
#  1. Raw EEG loading — directly from cnt_nback.mat (NO preprocessing)
# ══════════════════════════════════════════════════════════════════════════

def load_raw_eeg(subject_id: str):
    """
    Load raw EEG straight from cnt_nback.mat, with NO filtering, NO ICA,
    NO rereferencing, NO resampling — exactly as recorded by the device
    (except for the mandatory unit check the dataset itself documents).

    Returns
    -------
    data      : ndarray (n_channels, n_samples), unit: microvolts (uV)
    ch_names  : list[str]
    sfreq     : float (Hz, as recorded — NOT resampled to 200Hz)
    """
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    cnt_path   = os.path.join(eeg_folder, "cnt_nback.mat")

    if not os.path.exists(cnt_path):
        print(f"  [WARN] Raw EEG file not found: {cnt_path}")
        return None, None, None

    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_keys = [k for k in mat_cnt.keys() if not k.startswith("__")]
    cnt = mat_cnt[real_keys[0]]

    data     = np.asarray(cnt.x, dtype=np.float64).T   # (n_channels, n_samples)
    sfreq    = float(cnt.fs)
    ch_names = [str(ch).strip() for ch in cnt.clab]

    # The dataset stores EEG in Volts if values are small (<1.0); convert to
    # microvolts for consistency with the preprocessed-data script's
    # thresholds. This mirrors the same unit check the original
    # preprocessing pipeline performs on load.
    if np.max(np.abs(data)) < 1.0:
        data = data * 1e6   # Volt -> microvolt
    # otherwise assume the file already stores microvolts

    return data.astype(np.float32), ch_names, sfreq


# ══════════════════════════════════════════════════════════════════════════
#  2. Raw fNIRS loading — directly from cnt_nback.mat (NO preprocessing)
# ══════════════════════════════════════════════════════════════════════════

def load_raw_fnirs(subject_id: str):
    """
    Load raw fNIRS HbO/HbR straight from cnt_nback.mat, with NO filtering,
    NO baseline correction, NO wavelet/MARA cleaning, NO resampling.

    Note: the dataset descriptor (Shin et al. 2018) states that oxy/deoxy
    fields already represent hemoglobin CONCENTRATION changes (i.e.
    Beer-Lambert law was already applied by the data providers before
    distribution) — so "raw" here means raw HbO/HbR, not raw optical
    intensity, which is not available in this public dataset.

    Returns
    -------
    hbo, hbr : ndarray (n_channels, n_samples)
    sfreq    : float (Hz, as recorded)
    """
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")
    cnt_path     = os.path.join(fnirs_folder, "cnt_nback.mat")

    if not os.path.exists(cnt_path):
        print(f"  [WARN] Raw fNIRS file not found: {cnt_path}")
        return None, None, None

    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_keys = [k for k in mat_cnt.keys() if not k.startswith("__")]
    cnt = mat_cnt[real_keys[0]]

    oxy_raw   = np.asarray(cnt.oxy.x,   dtype=np.float64)   # (n_samples, n_ch)
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)   # (n_samples, n_ch)
    sfreq     = float(cnt.oxy.fs) if hasattr(cnt.oxy, "fs") else 10.0

    hbo = oxy_raw.T     # (n_ch, n_samples)
    hbr = deoxy_raw.T   # (n_ch, n_samples)

    return hbo.astype(np.float32), hbr.astype(np.float32), sfreq


# ══════════════════════════════════════════════════════════════════════════
#  3. EEG artifact detection (same methodology as the preprocessed-data script)
# ══════════════════════════════════════════════════════════════════════════

def bandpower(x: np.ndarray, sfreq: float, band: tuple) -> float:
    f, pxx = welch(x, fs=sfreq, nperseg=min(512, len(x)))
    idx = (f >= band[0]) & (f <= band[1])
    return float(np.trapz(pxx[idx], f[idx]))


def analyze_raw_eeg(data: np.ndarray, ch_names: list, sfreq: float) -> dict:
    """
    Run the core artifact checks (Z-score, kurtosis, PTP amplitude,
    powerline, EMG, flat-channel) on raw EEG. Uses the same fraction-based
    flagging fix validated on the preprocessed data.
    """
    n_ch, n_t = data.shape
    r = {}

    # Z-score, fraction-based
    z = np.abs(stats.zscore(data, axis=1))
    frac_bad_z = np.mean(z > THRESH_ZSCORE, axis=1)
    r["zscore_bad_ch"] = [ch_names[i] for i in np.where(frac_bad_z > CONTAM_FRAC_THRESH)[0]]
    r["max_z"] = float(z.max())

    # kurtosis
    kurt = stats.kurtosis(data, axis=1)
    r["kurtosis_bad_ch"] = [ch_names[i] for i in np.where(np.abs(kurt) > THRESH_KURTOSIS)[0]]
    r["kurtosis_values"] = kurt.tolist()
    r["max_abs_kurt"] = float(np.abs(kurt).max())

    # peak-to-peak amplitude
    ptp = np.ptp(data, axis=1)
    r["ptp_bad_ch"] = [ch_names[i] for i in np.where(ptp > THRESH_AMP_UV)[0]]
    r["max_ptp_uv"] = float(ptp.max())

    # 50 Hz powerline noise
    powerline_ch = []
    for ch, sig in enumerate(data):
        f, pxx = welch(sig, fs=sfreq, nperseg=min(1024, len(sig)))
        idx_pl = np.argmin(np.abs(f - 50.0))
        idx_bg = (f > 45) & (f < 49)
        if not np.any(idx_bg):
            continue
        db_above = 10 * np.log10(pxx[idx_pl] / (np.mean(pxx[idx_bg]) + 1e-20))
        if db_above > 8.0:
            powerline_ch.append(ch_names[ch])
    r["powerline_bad_ch"] = powerline_ch

    # EMG (high/low band power ratio)
    emg_ch = []
    for ch, sig in enumerate(data):
        low_p  = bandpower(sig, sfreq, (1, 30))
        high_p = bandpower(sig, sfreq, (30, min(90, sfreq / 2 - 1)))
        ratio  = high_p / (low_p + 1e-10)
        if ratio > THRESH_EMG_RATIO:
            emg_ch.append(ch_names[ch])
    r["emg_bad_ch"] = emg_ch

    # flat channels
    win = int(1.0 * sfreq)
    flat_ch = []
    for ch, sig in enumerate(data):
        stds = [np.std(sig[i:i + win]) for i in range(0, len(sig) - win, win)]
        if stds and np.mean(stds) < 0.05:
            flat_ch.append(ch_names[ch])
    r["flat_ch"] = flat_ch

    # simple composite score for quick comparison (same weighting logic
    # as the preprocessed-data script's compute_subject_quality_score)
    score = 0.0
    score += (len(r["zscore_bad_ch"]) / n_ch) * 20
    score += 15 if r["max_abs_kurt"] > 10 else 0
    score += 10 if r["max_ptp_uv"] > THRESH_AMP_UV else 0
    score += 10 if r["powerline_bad_ch"] else 0
    score += 10 if r["emg_bad_ch"] else 0
    score += 15 if r["flat_ch"] else 0
    r["composite_score"] = round(min(score, 100), 1)

    return r


# ══════════════════════════════════════════════════════════════════════════
#  4. fNIRS artifact detection (same methodology as the preprocessed-data script)
# ══════════════════════════════════════════════════════════════════════════

def analyze_raw_fnirs(hbo: np.ndarray, hbr: np.ndarray, sfreq: float) -> dict:
    """
    Run core fNIRS artifact checks (SNR, kurtosis, drift, HbO-HbR
    correlation, motion spikes) on raw HbO/HbR, averaged across channels
    for a whole-recording summary (raw data here is multi-channel, unlike
    the reconstructed single-trace signal used for the preprocessed data).
    """
    n_ch = hbo.shape[0]
    r = {"n_channels": n_ch}

    snr_vals, kurt_vals, drift_vals, corr_vals, spike_counts = [], [], [], [], []

    for ch in range(n_ch):
        sig_hbo = hbo[ch]
        sig_hbr = hbr[ch]

        # SNR (low-freq signal power vs high-freq noise power)
        f, pxx = welch(sig_hbo, fs=sfreq, nperseg=min(256, len(sig_hbo) // 2))
        sig_p   = float(np.sum(pxx[f < 0.5]))
        noise_p = float(np.sum(pxx[(f >= 2.0) & (f <= 4.0)])) + 1e-10
        snr_vals.append(10 * np.log10(sig_p / noise_p))

        kurt_vals.append(float(stats.kurtosis(sig_hbo)))

        t = np.arange(len(sig_hbo))
        coef = np.polyfit(t, sig_hbo, 1)
        trend = np.polyval(coef, t)
        drift_vals.append(float(abs(trend[-1] - trend[0])))

        if np.std(sig_hbo) > 1e-8 and np.std(sig_hbr) > 1e-8:
            corr_vals.append(float(stats.pearsonr(sig_hbo, sig_hbr)[0]))

        deriv  = np.abs(np.diff(sig_hbo))
        thresh = np.mean(deriv) + 4.0 * np.std(deriv)
        idx, _ = find_peaks(deriv, height=thresh, distance=int(sfreq))
        spike_counts.append(len(idx))

    r["snr_db_mean"]       = float(np.mean(snr_vals))
    r["snr_db_per_ch"]     = snr_vals
    r["kurtosis_mean"]     = float(np.mean(kurt_vals))
    r["kurtosis_per_ch"]   = kurt_vals
    r["drift_mean"]        = float(np.mean(drift_vals))
    r["drift_per_ch"]      = drift_vals
    r["hbo_hbr_corr_mean"] = float(np.mean(corr_vals)) if corr_vals else 0.0
    r["motion_spikes_total"] = int(np.sum(spike_counts))
    r["motion_spikes_per_ch"] = spike_counts

    score = 0.0
    score += 15 if r["snr_db_mean"] < THRESH_FNIRS_SNR_DB else 0
    score += 10 if abs(r["kurtosis_mean"]) > THRESH_FNIRS_KURTOSIS else 0
    score += 10 if r["drift_mean"] > THRESH_FNIRS_DRIFT else 0
    score += 10 if r["hbo_hbr_corr_mean"] > THRESH_FNIRS_HBR_CORR else 0
    score += min(20, r["motion_spikes_total"])
    r["composite_score"] = round(min(score, 100), 1)

    return r


# ══════════════════════════════════════════════════════════════════════════
#  5. Visualization
# ══════════════════════════════════════════════════════════════════════════

def plot_raw_eeg(data: np.ndarray, ch_names: list, sfreq: float,
                 eeg_r: dict, subject_id: str, output_dir: str) -> str:
    t = np.arange(data.shape[1]) / sfreq
    bad_names = set(eeg_r["zscore_bad_ch"]) | set(eeg_r["kurtosis_bad_ch"])
    n_show = min(data.shape[0], 10)

    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(f"RAW (pre-preprocessing) EEG — {subject_id}\n"
                f"composite score={eeg_r['composite_score']}/100 "
                f"(n_channels={data.shape[0]}, sfreq={sfreq:.0f}Hz, "
                f"dur={data.shape[1]/sfreq:.0f}s)",
                fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

    ax1 = fig.add_subplot(gs[0, :])
    offsets = np.arange(n_show) * float(np.ptp(data[:n_show]) * 0.5 + 1)
    for i in range(n_show):
        color = "tomato" if ch_names[i] in bad_names else "steelblue"
        ax1.plot(t, data[i] + offsets[i], lw=0.3, color=color, alpha=0.8)
    ax1.set_yticks(offsets)
    ax1.set_yticklabels(ch_names[:n_show], fontsize=7)
    ax1.set_xlabel("Time (s)")
    ax1.set_title(f"Raw EEG — first {n_show} channels (red = flagged)", fontsize=10)

    ax2 = fig.add_subplot(gs[1, 0])
    for i, sig in enumerate(data):
        f, pxx = welch(sig, fs=sfreq, nperseg=min(512, len(sig)))
        color = "tomato" if ch_names[i] in bad_names else "steelblue"
        ax2.semilogy(f, pxx, lw=0.5, color=color, alpha=0.6)
    ax2.axvline(50, color="darkred", ls="--", lw=1.2, label="50 Hz")
    ax2.set_xlim(0, min(95, sfreq / 2))
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Power (uV^2/Hz)")
    ax2.set_title("PSD — all channels")
    ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(gs[1, 1])
    kurt_vals = np.abs(eeg_r["kurtosis_values"])
    colors_k = ["tomato" if k > THRESH_KURTOSIS else "steelblue" for k in kurt_vals]
    ax3.bar(range(len(kurt_vals)), kurt_vals, color=colors_k)
    ax3.axhline(THRESH_KURTOSIS, color="red", ls="--", label=f"thresh={THRESH_KURTOSIS}")
    ax3.set_xlabel("Channel index")
    ax3.set_ylabel("|Kurtosis|")
    ax3.set_title("Kurtosis per Channel (raw)")
    ax3.legend(fontsize=8)

    path = os.path.join(output_dir, f"{subject_id}_RAW_eeg_report.png")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    return path


def plot_raw_fnirs(hbo: np.ndarray, hbr: np.ndarray, sfreq: float,
                   fnirs_r: dict, subject_id: str, output_dir: str) -> str:
    t = np.arange(hbo.shape[1]) / sfreq
    n_show = min(hbo.shape[0], 4)

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(f"RAW (pre-preprocessing) fNIRS — {subject_id}\n"
                f"composite score={fnirs_r['composite_score']}/100 "
                f"(n_channels={hbo.shape[0]}, sfreq={sfreq:.0f}Hz, "
                f"dur={hbo.shape[1]/sfreq:.0f}s)",
                fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

    ax1 = fig.add_subplot(gs[0, :])
    offsets = np.arange(n_show) * float(np.ptp(hbo[:n_show]) * 0.6 + 2)
    for i in range(n_show):
        ax1.plot(t, hbo[i] + offsets[i], lw=0.4, color="#E05252", alpha=0.8,
                 label="HbO" if i == 0 else None)
        ax1.plot(t, hbr[i] + offsets[i], lw=0.4, color="#4A90D9", alpha=0.8,
                 label="HbR" if i == 0 else None)
    ax1.set_xlabel("Time (s)")
    ax1.set_title(f"Raw fNIRS — first {n_show} channels", fontsize=10)
    ax1.legend(fontsize=8)

    ax2 = fig.add_subplot(gs[1, 0])
    ax2.hist(fnirs_r["snr_db_per_ch"], bins=15, color="steelblue", edgecolor="white")
    ax2.axvline(THRESH_FNIRS_SNR_DB, color="red", ls="--", label="thresh")
    ax2.set_xlabel("SNR (dB)")
    ax2.set_title("SNR distribution across channels (raw HbO)")
    ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(gs[1, 1])
    ax3.bar(range(len(fnirs_r["motion_spikes_per_ch"])),
            fnirs_r["motion_spikes_per_ch"], color="tomato")
    ax3.set_xlabel("Channel index")
    ax3.set_ylabel("# spikes")
    ax3.set_title("Motion spikes per channel (raw HbO)")

    path = os.path.join(output_dir, f"{subject_id}_RAW_fnirs_report.png")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    return path


# ══════════════════════════════════════════════════════════════════════════
#  6. Text reports
# ══════════════════════════════════════════════════════════════════════════

def write_eeg_text_report(eeg_r: dict, subject_id: str, output_dir: str) -> str:
    lines = [
        "=" * 60,
        f"RAW EEG Artifact Report — {subject_id}  (BEFORE preprocessing)",
        "=" * 60,
        f"Composite score: {eeg_r['composite_score']}/100  (higher = worse)",
        "",
        f"Z-score flagged channels    ({len(eeg_r['zscore_bad_ch'])}): {eeg_r['zscore_bad_ch']}",
        f"Kurtosis flagged channels   ({len(eeg_r['kurtosis_bad_ch'])}): {eeg_r['kurtosis_bad_ch']}  (max={eeg_r['max_abs_kurt']:.2f})",
        f"PTP-amplitude flagged       ({len(eeg_r['ptp_bad_ch'])}): {eeg_r['ptp_bad_ch']}  (max={eeg_r['max_ptp_uv']:.1f} uV)",
        f"50Hz powerline flagged      ({len(eeg_r['powerline_bad_ch'])}): {eeg_r['powerline_bad_ch']}",
        f"EMG (muscle) flagged        ({len(eeg_r['emg_bad_ch'])}): {eeg_r['emg_bad_ch']}",
        f"Flat channels               ({len(eeg_r['flat_ch'])}): {eeg_r['flat_ch']}",
        "=" * 60,
    ]
    path = os.path.join(output_dir, f"{subject_id}_RAW_eeg_report.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path


def write_fnirs_text_report(fnirs_r: dict, subject_id: str, output_dir: str) -> str:
    lines = [
        "=" * 60,
        f"RAW fNIRS Artifact Report — {subject_id}  (BEFORE preprocessing)",
        "=" * 60,
        f"Composite score: {fnirs_r['composite_score']}/100  (higher = worse)",
        f"Channels: {fnirs_r['n_channels']}",
        "",
        f"Mean SNR (HbO)          : {fnirs_r['snr_db_mean']:.1f} dB",
        f"Mean |kurtosis| (HbO)   : {fnirs_r['kurtosis_mean']:.2f}",
        f"Mean drift (HbO)        : {fnirs_r['drift_mean']:.3f}",
        f"Mean HbO-HbR correlation: {fnirs_r['hbo_hbr_corr_mean']:.2f}  "
        f"(expected: negative, physiologically)",
        f"Total motion spikes     : {fnirs_r['motion_spikes_total']} (all channels combined)",
        "=" * 60,
    ]
    path = os.path.join(output_dir, f"{subject_id}_RAW_fnirs_report.txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path


# ══════════════════════════════════════════════════════════════════════════
#  7. Comparison with preprocessed-data results
# ══════════════════════════════════════════════════════════════════════════

def print_comparison(eeg_r: dict, fnirs_r: dict, subject_id: str):
    """
    Prints a side-by-side comparison against the preprocessed-data scores
    found earlier for this subject (hard-coded from the previous analysis
    reports — update these numbers if you re-run the preprocessed-data
    script and get different values).
    """
    # scores found in the preprocessed-data EEG/fNIRS reports for VP004
    preprocessed_eeg_scores   = {"S1": 80.5, "S2": 81.9, "S3": 82.8}
    preprocessed_fnirs_scores = {"S1": 20.0, "S2": 8.0,  "S3": 12.0}

    print("\n" + "=" * 70)
    print(f"  COMPARISON — {subject_id}: RAW vs PREPROCESSED artifact scores")
    print("=" * 70)
    print(f"\n  EEG:")
    print(f"    Raw (whole recording, before preprocessing): {eeg_r['composite_score']}/100")
    print(f"    Preprocessed (per session):                  {preprocessed_eeg_scores}")
    print(f"\n  fNIRS:")
    print(f"    Raw (whole recording, before preprocessing): {fnirs_r['composite_score']}/100")
    print(f"    Preprocessed (per session):                  {preprocessed_fnirs_scores}")
    print()
    if eeg_r["composite_score"] > 40:
        print("  -> EEG artifact is ALREADY present in the raw signal.")
        print("     This points to a genuine recording/physiological issue")
        print("     (electrode contact, muscle tension, etc.), NOT something")
        print("     introduced or amplified by the preprocessing pipeline.")
    else:
        print("  -> EEG artifact score is LOW in the raw signal but HIGH after")
        print("     preprocessing. This would suggest the preprocessing pipeline")
        print("     itself is introducing or amplifying the problem for this")
        print("     subject — worth a closer look at the EEG preprocessing steps.")
    print("=" * 70)


# ══════════════════════════════════════════════════════════════════════════
#  Entry point
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Analyzing RAW (pre-preprocessing) data for {SUBJECT_ID} ...")

    # ── EEG ──────────────────────────────────────────────────────────────
    print("\n[EEG]")
    eeg_data, eeg_ch_names, eeg_sfreq = load_raw_eeg(SUBJECT_ID)
    if eeg_data is not None:
        print(f"  Loaded: {eeg_data.shape[0]} channels x {eeg_data.shape[1]} samples "
              f"@ {eeg_sfreq:.0f} Hz ({eeg_data.shape[1]/eeg_sfreq:.0f}s)")
        eeg_r = analyze_raw_eeg(eeg_data, eeg_ch_names, eeg_sfreq)
        png = plot_raw_eeg(eeg_data, eeg_ch_names, eeg_sfreq, eeg_r, SUBJECT_ID, OUTPUT_DIR)
        txt = write_eeg_text_report(eeg_r, SUBJECT_ID, OUTPUT_DIR)
        print(f"  Composite score: {eeg_r['composite_score']}/100")
        print(f"  [saved] {png}")
        print(f"  [saved] {txt}")
    else:
        eeg_r = None
        print("  Could not load raw EEG — check BASE_DATASET_PATH.")

    # ── fNIRS ────────────────────────────────────────────────────────────
    print("\n[fNIRS]")
    hbo, hbr, fnirs_sfreq = load_raw_fnirs(SUBJECT_ID)
    if hbo is not None:
        print(f"  Loaded: {hbo.shape[0]} channels x {hbo.shape[1]} samples "
              f"@ {fnirs_sfreq:.0f} Hz ({hbo.shape[1]/fnirs_sfreq:.0f}s)")
        fnirs_r = analyze_raw_fnirs(hbo, hbr, fnirs_sfreq)
        png = plot_raw_fnirs(hbo, hbr, fnirs_sfreq, fnirs_r, SUBJECT_ID, OUTPUT_DIR)
        txt = write_fnirs_text_report(fnirs_r, SUBJECT_ID, OUTPUT_DIR)
        print(f"  Composite score: {fnirs_r['composite_score']}/100")
        print(f"  [saved] {png}")
        print(f"  [saved] {txt}")
    else:
        fnirs_r = None
        print("  Could not load raw fNIRS — check BASE_DATASET_PATH.")

    # ── comparison ───────────────────────────────────────────────────────
    if eeg_r is not None and fnirs_r is not None:
        print_comparison(eeg_r, fnirs_r, SUBJECT_ID)
